In [ ]:
import sys
import importlib

# Ensure utils are in path
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
UTILS_DIR = next(
    (candidate / "utils" for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (candidate / "utils").is_dir()),
    None,
)
if UTILS_DIR and str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

# Try to reload the modules
import two_stage_common
import regressor_tuning_common

importlib.reload(two_stage_common)
importlib.reload(regressor_tuning_common)

print("Modules reloaded.")
print(f"mlflow in two_stage_common: {two_stage_common.mlflow}")
print(f"mlflow in regressor_tuning_common: {regressor_tuning_common.mlflow}")

In [ ]:

# Investigating the Index/Columns dtype
from regressor_tuning_common import prepare_threshold_regression_data
import pandas as pd

data = prepare_threshold_regression_data(sample_frac=0.01)
X = data["X_search"]

print(f"Columns Index type: {type(X.columns)}")
print(f"Columns Index dtype: {X.columns.dtype}")
print(f"First 5 column names: {X.columns[:5].tolist()}")

# Check if any part of the index or metadata uses StringDtype
try:
    from pandas.api.types import is_string_dtype
    print(f"Is Columns Index string dtype? {is_string_dtype(X.columns)}")
except:
    pass

# Try to force columns to object
X.columns = X.columns.astype(object)
print(f"After conversion - Columns Index dtype: {X.columns.dtype}")

# Tuning - Random Forest Regressor para limiar de reposicao

Este notebook desenvolve o sucessor do artefato historico `03_tree_ensembles_random_forest_regressor_threshold_model.pkl`, mas sem salvar `.pkl` localmente. O modelo, metricas, predicoes, parametros e graficos sao registrados diretamente no MLflow.


## Estrategia

Usamos `TimeSeriesSplit`, uma validacao cruzada temporal. Ela preserva a ordem passado -> futuro e evita vazamento temporal, que ocorreria com K-Fold aleatorio. O conjunto `test` fica isolado para avaliacao final do campeao.

O tuning usa `RandomizedSearchCV` com uma configuracao leve, porque o grid completo seria caro para florestas com varias combinacoes de profundidade, folhas e numero de arvores. A busca avalia poucos candidatos e apenas os perfis de peso principais para poupar CPU.


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
UTILS_DIR = next(
    (candidate / "utils" for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (candidate / "utils").is_dir()),
    None,
)
if UTILS_DIR is None:
    raise RuntimeError("Nao encontrei ml/notebooks/utils. Execute o notebook dentro do repositorio Saltim.")
sys.path.insert(0, str(UTILS_DIR))

import pandas as pd

from regressor_tuning_common import (
    RegressorTuningConfig,
    WEIGHT_PROFILE_DESCRIPTIONS,
    run_regressor_tuning,
)

from sklearn.ensemble import RandomForestRegressor


In [ ]:
RF_PARAM_DISTRIBUTIONS = {
    "model__n_estimators": [80, 120, 180],
    "model__max_depth": [6, 10, 14],
    "model__min_samples_split": [5, 10, 20],
    "model__min_samples_leaf": [2, 4, 8],
    "model__max_features": ["sqrt", 0.5, 0.8],
    "model__bootstrap": [True],
}

def rf_model_factory() -> RandomForestRegressor:
    return RandomForestRegressor(random_state=42, n_jobs=2)

def rf_baseline_factory() -> RandomForestRegressor:
    return RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=2)

config = RegressorTuningConfig(
    notebook_id="05_random_forest_regressor_threshold_tuning",
    family="Arvores",
    model_name="Random Forest Regressor",
    model_factory=rf_model_factory,
    baseline_factory=rf_baseline_factory,
    param_distributions=RF_PARAM_DISTRIBUTIONS,
    n_iter=6,
    cv_splits=3,
    random_state=42,
    n_jobs=1,
    use_full_dataset=True,
    weight_profiles=("uniform", "alert_focus"),
)

search_space = pd.DataFrame(
    [{"parametro": key, "valores": values} for key, values in RF_PARAM_DISTRIBUTIONS.items()]
)
weight_profiles = pd.DataFrame(
    [
        {"perfil": key, "descricao": WEIGHT_PROFILE_DESCRIPTIONS[key]}
        for key in config.weight_profiles
    ]
)

display(search_space)
display(weight_profiles)


## Execucao do tuning

A celula abaixo executa baseline, buscas randomicas por perfil de peso, registro dos candidatos no MLflow, treino final do campeao e diagnosticos de ajuste. A configuracao foi reduzida para poupar CPU: poucos candidatos, 3 folds, dois perfis de peso e paralelismo limitado.


In [ ]:
results = run_regressor_tuning(config)

## Resultados quantitativos

As tabelas abaixo mostram baseline, melhores candidatos, variancia entre folds e metricas finais no teste.


In [ ]:
display(results["baseline_summary"])
display(results["candidate_results"].head(15))
display(results["best_fold_metrics"])
display(results["final_metrics"].T)
print("Melhor perfil de peso:", results["best_weight_profile"])
print("Melhores hiperparametros:", results["best_params"])
print("Diagnostico:", results["fit_diagnosis"])


## Visualizacoes de ajuste

A curva de aprendizado ajuda a identificar overfitting ou underfitting. A analise de residuos mostra vieses e dispersao dos erros no limiar previsto.


In [ ]:
results["figures"]["learning_curve"]


In [ ]:
results["figures"]["residual_analysis"]


## MLflow

No MLflow, procure pelos runs do experimento `notebooks/02_modelos_finais/05_random_forest_regressor_threshold_tuning/threshold_regression`. Os runs de candidatos guardam combinacoes de parametros e pesos; o run `champion` guarda o modelo registrado, predicoes, metricas finais e graficos.
